# Notebook 26 — Residual Diffusion Dynamics and Universality Persistence

This notebook extends the residual universality manifold into a dynamic diffusion system.

Previous notebooks established:

- Notebook 24: continuous residual universality density fields.
- Notebook 25: geodesic transport and bridgeability over the residual manifold.

Notebook 26 asks:

> Do residual universality regions persist under diffusion, or collapse into mixed transport states?

The notebook is self-contained. It first tries to load prior outputs from `results/`. If those files are unavailable in Colab, it generates a deterministic synthetic fallback manifold so every cell runs from a clean runtime.


In [ ]:
# Optional: mount Google Drive if you keep the repo there.
# from google.colab import drive
# drive.mount('/content/drive')


In [ ]:
from pathlib import Path
import json
import zipfile
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import kneighbors_graph
from sklearn.metrics import pairwise_distances
from sklearn.decomposition import PCA

try:
    from scipy.linalg import expm
    from scipy.sparse.csgraph import connected_components
    from scipy.spatial.distance import pdist, squareform
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False

warnings.filterwarnings('ignore')

RANDOM_SEED = 9423
rng = np.random.default_rng(RANDOM_SEED)

# Directory convention: repo root has notebooks/, figures/, results/, exports/.
# In Colab opened directly from GitHub, cwd is usually /content.
ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent

FIGURES_DIR = ROOT / 'figures'
RESULTS_DIR = ROOT / 'results'
EXPORTS_DIR = ROOT / 'exports'
for d in [FIGURES_DIR, RESULTS_DIR, EXPORTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('ROOT:', ROOT)
print('FIGURES_DIR:', FIGURES_DIR)
print('RESULTS_DIR:', RESULTS_DIR)
print('EXPORTS_DIR:', EXPORTS_DIR)
print('available results:', sorted(p.name for p in RESULTS_DIR.glob('*'))[:40])


## 1. Load residual manifold inputs or build deterministic fallback

The loader searches for the same outputs used by recent notebooks. If none are present, the fallback creates five residual topology families at four graph sizes.


In [ ]:
def read_first_existing(paths):
    for p in paths:
        p = Path(p)
        if p.exists():
            print(f'loaded: {p}')
            return pd.read_csv(p), p
    return None, None

candidates = [
    RESULTS_DIR / 'residual_universality_embedding.csv',
    RESULTS_DIR / 'residual_pca_embedding.csv',
    RESULTS_DIR / 'residual_classification_feature_matrix.csv',
    RESULTS_DIR / 'residual_geometry_features.csv',
    RESULTS_DIR / '20_residual_universality_embedding.csv',
    RESULTS_DIR / '21_fixed_point_estimates.csv',
    RESULTS_DIR / '24_known_manifold_embedding.csv',
    RESULTS_DIR / '25_transport_nodes.csv',
]

raw, source_path = read_first_existing(candidates)
print('source_path:', source_path)


In [ ]:
def normalize_topology_name(x):
    s = str(x).strip()
    mapping = {
        'erdos renyi': 'Erdős–Rényi',
        'erdos-renyi': 'Erdős–Rényi',
        'erdos_renyi': 'Erdős–Rényi',
        'Erdos-Renyi': 'Erdős–Rényi',
        'Erdős-Rényi': 'Erdős–Rényi',
        'Erdős–Rényi': 'Erdős–Rényi',
        'ring_lattice': 'ring lattice',
        'ring lattice': 'ring lattice',
        'small_world': 'small world',
        'small world': 'small world',
        'scale_free': 'scale free',
        'scale free': 'scale free',
        'modular_clustered': 'modular clustered',
        'modular clustered': 'modular clustered',
    }
    return mapping.get(s, s)

TOPOLOGY_ORDER = ['ring lattice', 'small world', 'Erdős–Rényi', 'scale free', 'modular clustered']
SIZES = [16, 32, 64, 128]

def infer_columns(df):
    cols = list(df.columns)
    topo_col = next((c for c in cols if c.lower() in ['topology','family','label','known_topology','source_topology']), None)
    size_col = next((c for c in cols if c.lower() in ['n','size','graph_size','n_modules','N'.lower()]), None)
    pc1_col = next((c for c in cols if c.lower() in ['pc1','x','embedding_x','coordinate_1','residual manifold coordinate 1']), None)
    pc2_col = next((c for c in cols if c.lower() in ['pc2','y','embedding_y','coordinate_2','residual manifold coordinate 2']), None)
    return topo_col, size_col, pc1_col, pc2_col

if raw is not None:
    topo_col, size_col, pc1_col, pc2_col = infer_columns(raw)
    print('inferred columns:', topo_col, size_col, pc1_col, pc2_col)
else:
    topo_col = size_col = pc1_col = pc2_col = None


In [ ]:
def build_fallback_manifold(seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    base = {
        'ring lattice': (-1.3, 0.2),
        'small world': (-2.7, 1.1),
        'Erdős–Rényi': (-0.5, -0.2),
        'scale free': (1.4, -0.5),
        'modular clustered': (3.2, -0.15),
    }
    drift = {
        'ring lattice': (0.18, -0.55),
        'small world': (-0.45, 0.25),
        'Erdős–Rényi': (-0.25, -0.55),
        'scale free': (-0.32, -0.48),
        'modular clustered': (0.18, -0.22),
    }
    rows = []
    for topo in TOPOLOGY_ORDER:
        bx, by = base[topo]
        dx, dy = drift[topo]
        for i, N in enumerate(SIZES):
            t = i / (len(SIZES)-1)
            wobble = rng.normal(0, 0.08, 2)
            rows.append({
                'topology': topo,
                'N': N,
                'PC1': bx + dx*i + wobble[0],
                'PC2': by + dy*i + wobble[1],
                'source': 'synthetic_fallback',
            })
    return pd.DataFrame(rows)

if raw is not None and topo_col and pc1_col and pc2_col:
    manifold = pd.DataFrame({
        'topology': raw[topo_col].map(normalize_topology_name),
        'N': raw[size_col] if size_col else np.tile(SIZES, math.ceil(len(raw)/len(SIZES)))[:len(raw)],
        'PC1': pd.to_numeric(raw[pc1_col], errors='coerce'),
        'PC2': pd.to_numeric(raw[pc2_col], errors='coerce'),
    })
    manifold = manifold.dropna(subset=['topology','PC1','PC2']).copy()
    data_source = f'loaded {source_path.name}'
else:
    manifold = build_fallback_manifold()
    source_path = None
    data_source = 'deterministic synthetic fallback'

# Keep known families only where possible.
manifold['topology'] = manifold['topology'].map(normalize_topology_name)
if set(TOPOLOGY_ORDER).intersection(set(manifold['topology'])):
    manifold = manifold[manifold['topology'].isin(TOPOLOGY_ORDER)].copy()

manifold['N'] = pd.to_numeric(manifold['N'], errors='coerce')
if manifold['N'].isna().any():
    manifold['N'] = np.tile(SIZES, math.ceil(len(manifold)/len(SIZES)))[:len(manifold)]
manifold['N'] = manifold['N'].astype(int)

# Standardize orientation only if needed: keep values as passed, but add normalized coordinates for graph operations.
coords = manifold[['PC1','PC2']].to_numpy(float)
scaler = StandardScaler()
scaled = scaler.fit_transform(coords)
manifold['X1'] = scaled[:,0]
manifold['X2'] = scaled[:,1]

out_path = RESULTS_DIR / '26_manifold_nodes.csv'
manifold.to_csv(out_path, index=False)
print(data_source)
print('nodes:', len(manifold))
print(manifold.head())


## 2. Build weighted kNN diffusion graph

The graph uses residual manifold coordinates and Gaussian edge weights. It is symmetric and row-normalized for diffusion.


In [ ]:
def build_weighted_knn_graph(points, k=4, sigma=None):
    n = len(points)
    k = max(1, min(k, n-1))
    D = pairwise_distances(points)
    if sigma is None:
        sigma = np.median(D[D > 0]) if np.any(D > 0) else 1.0
    A = np.zeros((n,n), dtype=float)
    for i in range(n):
        idx = np.argsort(D[i])[1:k+1]
        for j in idx:
            w = np.exp(-(D[i,j]**2) / (2*sigma**2 + 1e-12))
            A[i,j] = max(A[i,j], w)
            A[j,i] = max(A[j,i], w)
    return A, D, sigma

points = manifold[['X1','X2']].to_numpy(float)
A, Dmat, sigma = build_weighted_knn_graph(points, k=4)
deg = A.sum(axis=1)
L = np.diag(deg) - A
P = A / np.maximum(deg[:,None], 1e-12)

if SCIPY_AVAILABLE:
    n_components, labels = connected_components(A > 0, directed=False)
else:
    n_components, labels = (np.nan, np.zeros(len(A), dtype=int))

edge_rows = []
for i in range(len(A)):
    for j in range(i+1, len(A)):
        if A[i,j] > 0:
            edge_rows.append({
                'i': i,
                'j': j,
                'source_topology': manifold.iloc[i]['topology'],
                'target_topology': manifold.iloc[j]['topology'],
                'source_N': int(manifold.iloc[i]['N']),
                'target_N': int(manifold.iloc[j]['N']),
                'weight': A[i,j],
                'distance': Dmat[i,j],
            })
edges = pd.DataFrame(edge_rows)
edges.to_csv(RESULTS_DIR / '26_knn_diffusion_edges.csv', index=False)
np.save(RESULTS_DIR / '26_adjacency.npy', A)
np.save(RESULTS_DIR / '26_laplacian.npy', L)

print('sigma:', sigma)
print('connected components:', n_components)
print('edges:', len(edges))
edges.head()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
for _, e in edges.iterrows():
    i, j = int(e['i']), int(e['j'])
    ax.plot([manifold.iloc[i]['PC1'], manifold.iloc[j]['PC1']],
            [manifold.iloc[i]['PC2'], manifold.iloc[j]['PC2']],
            linewidth=0.4 + 2.0*e['weight'], alpha=0.25, color='black')
for topo in TOPOLOGY_ORDER:
    sub = manifold[manifold['topology'] == topo].sort_values('N')
    ax.plot(sub['PC1'], sub['PC2'], marker='o', linewidth=2, label=topo)
    for _, r in sub.iterrows():
        ax.text(r['PC1'], r['PC2'], f"N={int(r['N'])}", fontsize=8)
ax.axhline(0, linestyle='--', linewidth=1, color='gray')
ax.axvline(0, linestyle='--', linewidth=1, color='gray')
ax.set_title('Residual diffusion kNN graph')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_diffusion_knn_graph.png', dpi=180)
plt.show()


## 3. Heat diffusion dynamics

Each topology begins with mass distributed over its family nodes. Heat diffusion then propagates through the weighted graph.


In [ ]:
def transition_matrix(A):
    deg = A.sum(axis=1)
    return A / np.maximum(deg[:,None], 1e-12)

P = transition_matrix(A)
steps = list(range(0, 31))
alpha = 0.22  # lazy diffusion step size
T = (1-alpha)*np.eye(len(P)) + alpha*P

traj_rows = []
entropy_rows = []
persistence_rows = []

for topo in TOPOLOGY_ORDER:
    mask = (manifold['topology'] == topo).to_numpy()
    if not mask.any():
        continue
    u = np.zeros(len(manifold), dtype=float)
    u[mask] = 1.0 / mask.sum()
    for t in steps:
        entropy = -sum(float(p) * math.log(float(p) + 1e-12) for p in u if p > 0)
        persistence = float(u[mask].sum())
        entropy_rows.append({'topology': topo, 'step': t, 'entropy': entropy})
        persistence_rows.append({'topology': topo, 'step': t, 'universality_persistence': persistence})
        for idx, mass in enumerate(u):
            if mass > 1e-8:
                traj_rows.append({
                    'seed_topology': topo,
                    'step': t,
                    'node': idx,
                    'node_topology': manifold.iloc[idx]['topology'],
                    'N': int(manifold.iloc[idx]['N']),
                    'mass': float(mass),
                    'PC1': float(manifold.iloc[idx]['PC1']),
                    'PC2': float(manifold.iloc[idx]['PC2']),
                })
        u = T @ u
        u = u / max(u.sum(), 1e-12)

diffusion_traj = pd.DataFrame(traj_rows)
entropy_df = pd.DataFrame(entropy_rows)
persistence_df = pd.DataFrame(persistence_rows)

diffusion_traj.to_csv(RESULTS_DIR / '26_diffusion_trajectories.csv', index=False)
entropy_df.to_csv(RESULTS_DIR / '26_entropy_curves.csv', index=False)
persistence_df.to_csv(RESULTS_DIR / '26_universality_persistence.csv', index=False)

print(diffusion_traj.head())


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for topo in TOPOLOGY_ORDER:
    sub = entropy_df[entropy_df['topology'] == topo]
    axes[0].plot(sub['step'], sub['entropy'], marker='o', label=topo)
    sub = persistence_df[persistence_df['topology'] == topo]
    axes[1].plot(sub['step'], sub['universality_persistence'], marker='o', label=topo)

axes[0].set_title('Diffusion entropy growth')
axes[0].set_xlabel('diffusion step')
axes[0].set_ylabel('entropy')
axes[1].set_title('Universality persistence decay')
axes[1].set_xlabel('diffusion step')
axes[1].set_ylabel('mass retained inside source family')
axes[1].set_ylim(0, 1.05)
axes[1].legend()
for ax in axes:
    ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_entropy_and_persistence.png', dpi=180)
plt.show()


In [ ]:
selected_steps = [0, 2, 5, 10, 20, 30]
seed_topology = 'ring lattice' if 'ring lattice' in manifold['topology'].values else manifold['topology'].iloc[0]
plot_df = diffusion_traj[(diffusion_traj['seed_topology'] == seed_topology) & (diffusion_traj['step'].isin(selected_steps))]

fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharex=True, sharey=True)
axes = axes.ravel()
for ax, step in zip(axes, selected_steps):
    sub_mass = plot_df[plot_df['step'] == step]
    ax.scatter(manifold['PC1'], manifold['PC2'], s=60, alpha=0.25)
    ax.scatter(sub_mass['PC1'], sub_mass['PC2'], s=800*sub_mass['mass']+20, alpha=0.75)
    ax.set_title(f'{seed_topology}: step {step}')
    ax.axhline(0, linestyle='--', linewidth=0.8, color='gray')
    ax.axvline(0, linestyle='--', linewidth=0.8, color='gray')
    ax.grid(True, alpha=0.25)
fig.suptitle('Heat diffusion snapshots over residual manifold')
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_heat_diffusion_snapshots.png', dpi=180)
plt.show()


## 4. Diffusion phase transition sweep

Sweep graph sparsity `k` and diffusion step size `alpha`. Persistence is measured after a fixed number of steps.


In [ ]:
phase_rows = []
ks = [2, 3, 4, 5, 6, 8]
alphas = np.linspace(0.05, 0.65, 13)
final_step = 20

for k in ks:
    A_k, _, _ = build_weighted_knn_graph(points, k=k)
    P_k = transition_matrix(A_k)
    for alpha_val in alphas:
        T_k = (1-alpha_val)*np.eye(len(P_k)) + alpha_val*P_k
        vals = []
        for topo in TOPOLOGY_ORDER:
            mask = (manifold['topology'] == topo).to_numpy()
            if not mask.any():
                continue
            u = np.zeros(len(manifold)); u[mask] = 1.0 / mask.sum()
            for _ in range(final_step):
                u = T_k @ u
                u = u / max(u.sum(), 1e-12)
            vals.append(float(u[mask].sum()))
        phase_rows.append({
            'k': k,
            'alpha': float(alpha_val),
            'mean_persistence': float(np.mean(vals)),
            'min_persistence': float(np.min(vals)),
            'mixing': float(1 - np.mean(vals)),
        })

phase_df = pd.DataFrame(phase_rows)
phase_df.to_csv(RESULTS_DIR / '26_phase_transition_grid.csv', index=False)
phase_df.head()


In [ ]:
pivot = phase_df.pivot(index='k', columns='alpha', values='mean_persistence')
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(pivot.values, aspect='auto', origin='lower', vmin=0, vmax=1)
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f'{x:.2f}' for x in pivot.columns], rotation=45)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel('diffusion step size alpha')
ax.set_ylabel('kNN graph sparsity k')
ax.set_title('Universality persistence phase diagram')
fig.colorbar(im, ax=ax, label='mean persistence after 20 steps')
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_phase_transition_diagram.png', dpi=180)
plt.show()


## 5. Laplacian spectrum and transport harmonics

Low-frequency Laplacian modes identify global transport harmonics and weakly connected universality regions.


In [ ]:
evals, evecs = np.linalg.eigh(L)
spectrum_df = pd.DataFrame({'mode': np.arange(len(evals)), 'eigenvalue': evals})
spectrum_df.to_csv(RESULTS_DIR / '26_laplacian_spectrum.csv', index=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(spectrum_df['mode'], spectrum_df['eigenvalue'], marker='o')
ax.set_title('Residual diffusion Laplacian spectrum')
ax.set_xlabel('mode')
ax.set_ylabel('eigenvalue')
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_laplacian_spectrum.png', dpi=180)
plt.show()


In [ ]:
modes_to_plot = [1, 2, 3, 4]
fig, axes = plt.subplots(2, 2, figsize=(12, 10), sharex=True, sharey=True)
axes = axes.ravel()
for ax, mode in zip(axes, modes_to_plot):
    vals = evecs[:, mode] if mode < evecs.shape[1] else evecs[:, -1]
    sc = ax.scatter(manifold['PC1'], manifold['PC2'], c=vals, s=120, cmap='coolwarm')
    for _, r in manifold.iterrows():
        ax.text(r['PC1'], r['PC2'], f"{r['topology'][:3]} {int(r['N'])}", fontsize=7)
    ax.axhline(0, linestyle='--', linewidth=0.8, color='gray')
    ax.axvline(0, linestyle='--', linewidth=0.8, color='gray')
    ax.set_title(f'Laplacian eigenmode {mode}')
    ax.grid(True, alpha=0.25)
    fig.colorbar(sc, ax=ax, shrink=0.8)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_spectral_modes.png', dpi=180)
plt.show()


## 6. Diffusion bottlenecks and conductance proxy

A bottleneck edge combines long distance and low edge weight. This highlights fragile transport bridges.


In [ ]:
if len(edges):
    bottleneck_df = edges.copy()
    bottleneck_df['bottleneck_score'] = bottleneck_df['distance'] / (bottleneck_df['weight'] + 1e-9)
    bottleneck_df = bottleneck_df.sort_values('bottleneck_score', ascending=False)
else:
    bottleneck_df = pd.DataFrame(columns=['bottleneck_score'])

bottleneck_df.to_csv(RESULTS_DIR / '26_diffusion_bottlenecks.csv', index=False)
bottleneck_df.head(10)


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
if len(edges):
    scores = edges.copy()
    scores['bottleneck_score'] = scores['distance'] / (scores['weight'] + 1e-9)
    max_score = scores['bottleneck_score'].max()
    for _, e in scores.iterrows():
        i, j = int(e['i']), int(e['j'])
        lw = 0.5 + 4*(e['bottleneck_score']/max_score)
        alpha_e = 0.15 + 0.65*(e['bottleneck_score']/max_score)
        ax.plot([manifold.iloc[i]['PC1'], manifold.iloc[j]['PC1']],
                [manifold.iloc[i]['PC2'], manifold.iloc[j]['PC2']],
                linewidth=lw, alpha=alpha_e, color='black')
for topo in TOPOLOGY_ORDER:
    sub = manifold[manifold['topology'] == topo].sort_values('N')
    ax.plot(sub['PC1'], sub['PC2'], marker='o', linewidth=2, label=topo)
ax.axhline(0, linestyle='--', linewidth=1, color='gray')
ax.axvline(0, linestyle='--', linewidth=1, color='gray')
ax.set_title('Diffusion bottleneck map')
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.legend()
ax.grid(True, alpha=0.25)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_diffusion_bottleneck_map.png', dpi=180)
plt.show()


## 7. Universality collapse robustness sweep

Perturb graph edges and measure persistence after diffusion. Larger perturbation implies more random bridge edges and weaker original topology structure.


In [ ]:
def perturb_adjacency(A, noise_level, rng):
    n = A.shape[0]
    B = A.copy()
    # Randomly add weak bridge noise; preserve symmetry.
    scale = np.mean(A[A > 0]) if np.any(A > 0) else 0.1
    noise = rng.random((n,n))
    mask = rng.random((n,n)) < noise_level
    add = noise * mask * scale * noise_level
    add = np.triu(add, 1)
    add = add + add.T
    B = np.maximum(B, add)
    np.fill_diagonal(B, 0)
    return B

collapse_rows = []
noise_levels = np.linspace(0, 0.55, 12)
reps = 20
for eps in noise_levels:
    vals = []
    for rep in range(reps):
        B = perturb_adjacency(A, eps, rng)
        P_b = transition_matrix(B)
        T_b = (1-alpha)*np.eye(len(P_b)) + alpha*P_b
        topo_vals = []
        for topo in TOPOLOGY_ORDER:
            mask = (manifold['topology'] == topo).to_numpy()
            if not mask.any():
                continue
            u = np.zeros(len(manifold)); u[mask] = 1.0 / mask.sum()
            for _ in range(20):
                u = T_b @ u
                u = u / max(u.sum(), 1e-12)
            topo_vals.append(float(u[mask].sum()))
        vals.append(np.mean(topo_vals))
    collapse_rows.append({
        'noise_level': float(eps),
        'mean_persistence': float(np.mean(vals)),
        'std_persistence': float(np.std(vals)),
    })

collapse_df = pd.DataFrame(collapse_rows)
collapse_df.to_csv(RESULTS_DIR / '26_collapse_robustness.csv', index=False)
collapse_df.head()


In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.errorbar(collapse_df['noise_level'], collapse_df['mean_persistence'],
            yerr=collapse_df['std_persistence'], marker='o', capsize=3)
ax.set_title('Universality collapse robustness sweep')
ax.set_xlabel('random bridge perturbation level')
ax.set_ylabel('mean persistence after diffusion')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig(FIGURES_DIR / '26_universality_collapse_robustness.png', dpi=180)
plt.show()


## 8. Summary table

This table collects final persistence, entropy, spectral gap, bottleneck, and robustness indicators.


In [ ]:
final_persistence = persistence_df[persistence_df['step'] == max(steps)].groupby('topology')['universality_persistence'].mean()
final_entropy = entropy_df[entropy_df['step'] == max(steps)].groupby('topology')['entropy'].mean()

summary_rows = []
for topo in TOPOLOGY_ORDER:
    sub = manifold[manifold['topology'] == topo]
    if sub.empty:
        continue
    node_idx = sub.index.to_list()
    local_edges = edges[(edges['source_topology'] == topo) | (edges['target_topology'] == topo)] if len(edges) else pd.DataFrame()
    summary_rows.append({
        'topology': topo,
        'n_nodes': len(sub),
        'final_persistence': float(final_persistence.get(topo, np.nan)),
        'final_entropy': float(final_entropy.get(topo, np.nan)),
        'mean_degree': float(deg[node_idx].mean()),
        'max_bottleneck_score': float(local_edges.assign(score=local_edges['distance']/(local_edges['weight']+1e-9))['score'].max()) if len(local_edges) else np.nan,
    })

summary_df = pd.DataFrame(summary_rows)
summary_df['spectral_gap'] = float(evals[1]) if len(evals) > 1 else np.nan
summary_df['data_source'] = data_source
summary_df.to_csv(RESULTS_DIR / '26_diffusion_summary.csv', index=False)
summary_df


## 9. Export manifest and zip download

This cell matches the prior notebook pattern: it writes a manifest, bundles Notebook 26 figures/results, and optionally triggers Colab download.


In [ ]:
manifest = {
    'notebook': '26_residual_diffusion_and_universality_persistence.ipynb',
    'data_source': data_source,
    'source_file': str(source_path) if source_path is not None else None,
    'created_outputs': {
        'figures': sorted([p.name for p in FIGURES_DIR.glob('26_*.png')]),
        'results': sorted([p.name for p in RESULTS_DIR.glob('26_*')]),
    },
}

manifest_path = EXPORTS_DIR / '26_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2))

zip_path = EXPORTS_DIR / '26_residual_diffusion_and_universality_persistence_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in FIGURES_DIR.glob('26_*.png'):
        z.write(p, arcname=f'figures/{p.name}')
    for p in RESULTS_DIR.glob('26_*'):
        z.write(p, arcname=f'results/{p.name}')
    z.write(manifest_path, arcname='exports/26_manifest.json')

print('Wrote:', zip_path)
print('Zip size MB:', zip_path.stat().st_size / 1e6)
print(json.dumps(manifest, indent=2)[:2000])

# Optional Colab download
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception as e:
    print('Colab download skipped. Download manually from:', zip_path)
